# Taller 1 — Consumo Automatizado de APIs

**Asignatura:** MLY1101 — Machine Learning  
**Nombre del estudiante:** Hernán Lippke  
**Sección:** MLY1101_001V  
**Fecha:** 2026-08-15

## Pregunta u objetivo

> ¿Qué información puede recopilarse sobre los países del mundo desde fuentes geográfico-demográficas, económicas y culturales?

Las tres fuentes aportan a esta pregunta desde ángulos distintos. **No se busca responder la pregunta ni integrar, unir o cruzar las fuentes**: cada API se consume y se guarda de forma independiente. Las 3 APIs son **sin API key**, por lo que el notebook se reproduce con *Ejecutar todas* sin intervención manual.

## Consideraciones generales

- Debe utilizar **3 APIs diferentes** disponibles en: https://github.com/public-apis/public-apis
- Cada API debe aportar información relacionada con el mismo objetivo.
- Debe obtener **mínimo 200 registros por API**, salvo que la fuente disponga de menos registros en total.
- Cada API debe generar un archivo independiente en formato `.json`, `.xlsx`, `.csv` o `.txt`.
- **No realizar merge, join, concat ni cruces entre datasets.**
- El notebook debe poder ejecutarse nuevamente usando **Entorno de ejecución → Ejecutar todas**.


## 2. Fuentes seleccionadas

Tres APIs públicas de la lista `public-apis/public-apis`, todas **sin API key**:

| # | API | Aporta a la pregunta | Enlace |
|---|-----|----------------------|--------|
| 1 | **REST Countries** | Geografía y demografía (capital, región, población, área, idiomas, monedas, coordenadas) | https://restcountries.com/ |
| 2 | **World Bank** | Economía (indicador PIB per cápita por país) | https://data.worldbank.org/ |
| 3 | **Nager.Date** | Cultura (días festivos oficiales por país) | https://date.nager.at/ |

## 3. Configuración común

Imports, carpeta de salida y utilidades compartidas por las 3 descargas.

In [11]:
# Librerías base
import requests
import json
import time
import pandas as pd
from pathlib import Path

# Carpeta de salida: raíz del repositorio (los datasets se generan junto al notebook)
OUTPUT_DIR = Path('.')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Carpeta de salida: {OUTPUT_DIR.resolve()}')

Carpeta de salida: C:\Users\herna\OneDrive\Documentos\CARRERA INGENIERIA INFORMATICA DUOC\6TO SEMESTRE\MACHINE LEARNING\MLY1101-Taller1-LippkeHernan


## 3.1 API 1 — CountriesNow (demografía y geografía)

**Nombre de la API:** CountriesNow  
**Documentación:** https://countriesnow.space/  
**Endpoint utilizado:** https://countriesnow.space/api/v0.1/countries/population  
**Descripción de los datos:** población (histórico por año) de cada país; se toma el año más reciente.  
**Relación con el objetivo:** aporta la dimensión demográfica (población por país).  
**Autenticación:** ninguna (sin API key).

CountriesNow devuelve todos los países en una sola llamada, por lo que **no requiere paginación**. La paginación real de este trabajo se implementa en la **API 2 (World Bank)**, que sí entrega los datos por páginas.

In [15]:
# CONFIGURACIÓN API 1
API1_URL = 'https://countriesnow.space/api/v0.1/countries/population'
API1_MIN_REGISTROS = 200
api1_headers = {'User-Agent': 'TallerMLY1101-countriesnow/1.0'}
api1_params = {}

#====================================================================================================#

# CONSUMO API 1 — CountriesNow
# Una sola llamada devuelve todos los países. Se valida el PAYLOAD (campo 'error'),
# no solo el status, y se aplana cada país tomando el año más reciente de población.

def aplanar_poblacion(item):
    counts = item.get('populationCounts') or []
    ultimo = max(counts, key=lambda c: int(c.get('year', 0))) if counts else {}
    return {
        'pais': item.get('country'),
        'code': item.get('code'),
        'iso3': item.get('iso3'),
        'poblacion': ultimo.get('value'),
        'anio': ultimo.get('year'),
    }

api1_registros = []
response = requests.get(API1_URL, headers=api1_headers, params=api1_params, timeout=60)
print('Status API 1:', response.status_code)
response.raise_for_status()
data = response.json()

# Validación de contenido (no basta con el status 200).
if isinstance(data, dict) and data.get('error') is False and isinstance(data.get('data'), list):
    api1_registros = [aplanar_poblacion(p) for p in data['data']]
else:
    print('Payload inesperado:', str(data)[:200])

print('Registros API 1:', len(api1_registros))
pd.DataFrame(api1_registros)   # vista previa en pantalla

#====================================================================================================#

# GUARDAR DATASET API 1 (CSV)
API1_ARCHIVO = OUTPUT_DIR / 'dataset_api_1.csv'

pd.DataFrame(api1_registros).to_csv(API1_ARCHIVO, index=False, encoding='utf-8')

print('Archivo generado:', API1_ARCHIVO)
print('Registros guardados:', len(api1_registros))

Status API 1: 200
Registros API 1: 263
Archivo generado: dataset_api_1.csv
Registros guardados: 263


## 3.2 API 2 — World Bank (economía)

**Nombre de la API:** World Bank  
**Documentación:** https://data.worldbank.org/  
**Endpoint utilizado:** https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD  
**Descripción de los datos:** PIB per cápita (USD) por país para un año.  
**Relación con el objetivo:** aporta la dimensión económica.  
**Autenticación:** ninguna (sin API key).

In [16]:
# CONFIGURACIÓN API 2
API2_URL = 'https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD'
API2_MIN_REGISTROS = 200
API2_ANIO = '2022'
API2_PER_PAGE = 100
api2_headers = {'User-Agent': 'TallerMLY1101-worldbank/1.0'}

#====================================================================================================#

# CONSUMO API 2 — World Bank (CON PAGINACIÓN)
# World Bank responde una lista [metadata, [registros]]. La metadata trae 'pages';
# se recorren TODAS las páginas acumulando los registros -> paginación real.

def descargar_pagina_wb(page):
    params = {'format': 'json', 'per_page': API2_PER_PAGE, 'page': page, 'date': API2_ANIO}
    r = requests.get(API2_URL, params=params, headers=api2_headers, timeout=60)
    r.raise_for_status()
    d = r.json()
    if isinstance(d, list) and len(d) == 2 and isinstance(d[1], list):
        return d[0], d[1]
    return {}, []

def aplanar_wb(r):
    return {
        'pais': r.get('country', {}).get('value'),
        'iso3': r.get('countryiso3code'),
        'anio': r.get('date'),
        'pib_per_capita': r.get('value'),
    }

# Página 1: además de datos, indica cuántas páginas hay.
meta, primera = descargar_pagina_wb(1)
total_paginas = int(meta.get('pages', 1)) if meta else 1
print('Status API 2: 200 | total de páginas:', total_paginas)

crudos = list(primera)
for page in range(2, total_paginas + 1):
    _, pag = descargar_pagina_wb(page)
    crudos.extend(pag)
    time.sleep(1)
    print(f'  página {page}/{total_paginas} -> acumulado {len(crudos)}')

api2_registros = [aplanar_wb(r) for r in crudos]
print('Registros API 2:', len(api2_registros))
pd.DataFrame(api2_registros)   # vista previa en pantalla

#====================================================================================================#

# GUARDAR DATASET API 2 (CSV)
API2_ARCHIVO = OUTPUT_DIR / 'dataset_api_2.csv'

pd.DataFrame(api2_registros).to_csv(API2_ARCHIVO, index=False, encoding='utf-8')

print('Archivo generado:', API2_ARCHIVO)
print('Registros guardados:', len(api2_registros))

Status API 2: 200 | total de páginas: 3
  página 2/3 -> acumulado 200
  página 3/3 -> acumulado 265
Registros API 2: 265
Archivo generado: dataset_api_2.csv
Registros guardados: 265


## 3.3 API 3 — Nager.Date (cultura: días festivos)

**Nombre de la API:** Nager.Date  
**Documentación:** https://date.nager.at/  
**Endpoint utilizado:** https://date.nager.at/api/v3/AvailableCountries y .../PublicHolidays/{año}/{país}  
**Descripción de los datos:** días festivos oficiales por país.  
**Relación con el objetivo:** aporta la dimensión cultural.  
**Autenticación:** ninguna (sin API key, sin límite de tasa).

In [17]:
# CONFIGURACIÓN API 3
API3_BASE = 'https://date.nager.at/api/v3'
API3_MIN_REGISTROS = 200
API3_ANIO = 2024          # año fijo -> reproducible
API3_OBJETIVO = 300       # meta cómoda por encima de 200
api3_headers = {'User-Agent': 'TallerMLY1101-nager/1.0'}

#====================================================================================================#

# CONSUMO API 3 — Nager.Date
# Se listan los países disponibles y se recorre país por país pidiendo sus
# feriados del año fijo; se acumulan hasta superar la meta (iteración = paginación).

def get_json_lista(url):
    r = requests.get(url, headers=api3_headers, timeout=30)
    r.raise_for_status()
    d = r.json()
    return d if isinstance(d, list) else []

def aplanar_feriado(pais_nombre, h):
    tipos = h.get('types') or []
    return {
        'pais': pais_nombre,
        'countryCode': h.get('countryCode'),
        'fecha': h.get('date'),
        'nombre_local': h.get('localName'),
        'nombre_en': h.get('name'),
        'tipo': ', '.join(tipos) if tipos else None,
        'global': h.get('global'),
        'fijo': h.get('fixed'),
    }

paises = get_json_lista(f'{API3_BASE}/AvailableCountries')
print('Status API 3: 200 | países disponibles:', len(paises))

api3_registros = []
for p in paises:
    feriados = get_json_lista(f"{API3_BASE}/PublicHolidays/{API3_ANIO}/{p.get('countryCode')}")
    for h in feriados:
        api3_registros.append(aplanar_feriado(p.get('name'), h))
    time.sleep(0.2)
    if len(api3_registros) >= API3_OBJETIVO:
        break

print('Registros API 3:', len(api3_registros))
pd.DataFrame(api3_registros)   # vista previa en pantalla

#====================================================================================================#

# GUARDAR DATASET API 3 (CSV)
API3_ARCHIVO = OUTPUT_DIR / 'dataset_api_3.csv'

pd.DataFrame(api3_registros).to_csv(API3_ARCHIVO, index=False, encoding='utf-8')

print('Archivo generado:', API3_ARCHIVO)
print('Registros guardados:', len(api3_registros))

Status API 3: 200 | países disponibles: 204
Registros API 3: 310
Archivo generado: dataset_api_3.csv
Registros guardados: 310


# Resumen final

La siguiente celda debe ejecutarse al final y mostrar el resultado real de la carga.


In [18]:
total = len(api1_registros) + len(api2_registros) + len(api3_registros)

print('RESUMEN DE CARGA')
print('-' * 50)
print(f'API 1: {len(api1_registros)} registros - {API1_ARCHIVO.name}')
print(f'API 2: {len(api2_registros)} registros - {API2_ARCHIVO.name}')
print(f'API 3: {len(api3_registros)} registros - {API3_ARCHIVO.name}')
print('-' * 50)
print(f'TOTAL: {total} registros')

if len(api1_registros) < 200:
    print('ADVERTENCIA: API 1 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api2_registros) < 200:
    print('ADVERTENCIA: API 2 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api3_registros) < 200:
    print('ADVERTENCIA: API 3 tiene menos de 200 registros. Documente la excepción si corresponde.')


RESUMEN DE CARGA
--------------------------------------------------
API 1: 263 registros - dataset_api_1.csv
API 2: 265 registros - dataset_api_2.csv
API 3: 310 registros - dataset_api_3.csv
--------------------------------------------------
TOTAL: 838 registros
